<a href="https://colab.research.google.com/github/HeyOmkar07/flyrank-ml-internship-omkar/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HeyOmkar07/flyrank-ml-internship-omkar/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1 row = 1 client × 1 content item

In [ ]:

FEATURE_MONTH = "2026-02"
LABEL_MONTH = "2026-03"

print("Feature month:", FEATURE_MONTH)
print("Label month:", LABEL_MONTH)

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification

#### Features
The features are calculated only from the February 2026 feature window. They represent information that would have been available at the decision moment.

The five features used are:
1. February impressions
2. February clicks
3. February CTR
4. February average position
5. February performance trend

#### Label
The label is the March 2026 future outcome. I use `went_dark` as the binary outcome, where 1 means the content item had zero clicks in March and 0 means it received at least one click.

#### Context
`client_hash_id` and `content_hash_id` are context/identifier fields. They are required for grouping, joining and validation, but they are not treated as predictive model features.

#### Excluded
Future March performance fields are excluded from the feature set because they are not available at the February decision moment. They would introduce target leakage.

In [ ]:
from google.colab import userdata
import os

HF_TOKEN = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN

print("HF_TOKEN loaded:", HF_TOKEN is not None)

In [ ]:
!pip -q install datasets huggingface_hub

In [ ]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face authentication configured.")

In [ ]:
rel = "hf://datasets/FlyRank/internship-warehouse"

print(
    con.sql("""
        SELECT COUNT(*)
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
        )
    """).df()
)

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW fact_content_daily_performance AS
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
""")

In [ ]:
print(con.sql("SHOW TABLES").df())

In [ ]:
print(
    con.sql("""
        DESCRIBE fact_content_daily_performance
    """).df().to_string(index=False)
)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

client_hash_id + content_hash_id

In [ ]:
grain_check = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM fact_content_daily_performance
    WHERE report_date >= '2026-02-01'
      AND report_date <= '2026-02-28'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

grain_check

### Unit of analysis

The source table contains daily observations for each client-content combination. For this analysis, the final analytical unit is one `client_hash_id` × `content_hash_id` pair, with daily observations aggregated over the February 2026 feature window.

Therefore, the source grain is client × content × day, while the final feature-frame grain is client × content.

In [ ]:
date_check = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM fact_content_daily_performance
    WHERE report_date >= '2026-02-01'
      AND report_date <= '2026-02-28'
      AND gsc_data_available IS TRUE
""").df()

date_check

In [ ]:
availability_check = con.sql("""
    SELECT
        COUNT(*) AS available_rows
    FROM fact_content_daily_performance
    WHERE report_date >= '2026-02-01'
      AND report_date <= '2026-02-28'
      AND gsc_data_available IS TRUE
""").df()

availability_check

In [ ]:
feb_agg = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_feb,
        SUM(gsc_clicks) AS clk_feb,
        AVG(gsc_avg_position) AS avg_position_feb
    FROM fact_content_daily_performance
    WHERE report_date >= '2026-02-01'
      AND report_date <= '2026-02-28'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

feb_agg.head()

In [ ]:
feb_agg["ctr_feb"] = (
    feb_agg["clk_feb"] /
    feb_agg["imp_feb"].replace(0, float("nan"))
).fillna(0)

feb_agg.head()

February organic sessions

In [183]:
feb_agg = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_feb,
        SUM(gsc_clicks) AS clk_feb,
        AVG(gsc_avg_position) AS avg_position_feb,
        SUM(sessions_organic) AS organic_sessions_feb
    FROM fact_content_daily_performance
    WHERE report_date >= '2026-02-01'
      AND report_date <= '2026-02-28'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

feb_agg["ctr_feb"] = (
    feb_agg["clk_feb"] /
    feb_agg["imp_feb"].replace(0, float("nan"))
).fillna(0)

feb_agg.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,imp_feb,clk_feb,avg_position_feb,organic_sessions_feb,ctr_feb
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,0.0,0.000000
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,3.0,0.008186
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,0.0,0.000000
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,5.0,0.001024
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,1.0,0.002062


In [ ]:
mar_agg = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_mar,
        SUM(gsc_clicks) AS clk_mar
    FROM fact_content_daily_performance
    WHERE report_date >= '2026-03-01'
      AND report_date <= '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

mar_agg.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
frame = feb_agg.merge(
    mar_agg,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

frame[["imp_mar", "clk_mar"]] = frame[
    ["imp_mar", "clk_mar"]
].fillna(0)

frame.head()

In [ ]:
frame["went_dark"] = (
    frame["clk_mar"] == 0
).astype(int)

frame[
    [
        "client_hash_id",
        "content_hash_id",
        "imp_feb",
        "clk_feb",
        "ctr_feb",
        "avg_position_feb",
        "organic_sessions_feb",
        "went_dark"
    ]
].head()

In [ ]:
print("Total rows:", len(frame))
print("Went dark:", frame["went_dark"].sum())
print("Went dark rate:", frame["went_dark"].mean())

In [ ]:
feature_cols = [
    "imp_feb",
    "clk_feb",
    "ctr_feb",
    "avg_position_feb",
    "organic_sessions_feb"
]

X = frame[
    ["client_hash_id", "content_hash_id"] + feature_cols
].copy()

y = frame["went_dark"].copy()

X.head()

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Feature availability

| Feature | Available when? |
|---|---|
| `imp_feb` | Available after the February feature window because it is calculated only from February impressions. |
| `clk_feb` | Available after the February feature window because it is calculated only from February clicks. |
| `ctr_feb` | Available after the February feature window because it is calculated from February clicks and impressions. |
| `avg_position_feb` | Available after the February feature window because it is calculated only from February observations. |
| `organic_sessions_feb` | Available after the February feature window because it is calculated only from February organic sessions. |

All five predictive features are calculated exclusively from the February feature window. The March outcome is kept separate and is not used as a feature.

In [ ]:
frame["leaked_label"] = frame["went_dark"]

In [ ]:
frame[
    ["went_dark", "leaked_label"]
].head()

In [ ]:
from sklearn.metrics import roc_auc_score

leakage_score = roc_auc_score(
    frame["went_dark"],
    frame["leaked_label"]
)

print("Score with leakage:", leakage_score)

In [ ]:
frame = frame.drop(columns=["leaked_label"])

In [ ]:
print("leaked_label" in frame.columns)

### Deliberate leakage experiment

I intentionally created a `leaked_label` feature by copying the future `went_dark` label into the feature set.

The resulting score was artificially perfect because the feature directly contained the target value. This demonstrates target leakage: future outcome information must not be used as an input feature.

After demonstrating the effect, I removed `leaked_label`. The final feature set contains only February information that would have been available at the prediction decision point.

### Limitation

The analysis uses a monthly feature window and a monthly future outcome window. Therefore, short-term changes within a month may not be fully captured.

The label is based on observed Google Search performance and does not establish the external causes of a content item's performance change.

The future March window is kept separate from the February feature window to avoid temporal leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.